# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

# tips from Mark:
2 seconds delay to feel safe
No session, no cookies
User-Agent = project name, not browser

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [ ]:
# import the requests library (1 line)
import requests



# assign the root url (without /status) to the root_url variable for ease of reference (1 line)
root_url = "https://country-leaders.onrender.com"

# assign the /status endpoint to another variable called status_url (1 line)
status_url = "https://country-leaders.onrender.com/status"

# query the /status endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(status_url)

# check the status_code using a condition and print appropriate messages (4 lines)
if req.status_code == 200:
    print("Good to go")
else:
    print(status_url, req.status_code)


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [ ]:
# Set the countries_url variable (1 line)
countries_url = "https://country-leaders.onrender.com/countries"

# query the /countries endpoint using the get() method and store it in the req variable (1 line)
r = requests.get(countries_url)

# Get the JSON content and store it in the countries variable (1 line)
countries = r.json()

# display the request's status code and the countries variable (1 line)
print(r.status_code, countries)


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [ ]:
# Set the cookie_url variable (1 line)
cookies_url = "https://country-leaders.onrender.com/cookie"

# Query the enpoint, set the cookies variable and display it (2 lines)
cookies = requests.get(cookies_url)
print(cookies.json())

check = requests.get("https://country-leaders.onrender.com/check")
print(check.json())

# query the /countries endpoint, assign the output to the countries variable (1 line)
countries = requests.get(countries_url)

# display the countries variable (1 line)
print(countries.json())


## >>> 
## Can't do this the way the exercise asks for, the cookie expires immediately, even when both requests are made in the same cell.


Try to query the countries endpoint using the cookie, save the output and print it.

In [ ]:
# getting the countries
## Doing it as a session:
with requests.Session() as s:
    s1 = s.get(cookies_url)
    print(s1.json())
    
    ## checks the cookie
    check = s.get("https://country-leaders.onrender.com/check")
    print(check.json())
    
    s2 = s.get(countries_url)
    countries = s2.json()
    print(s2.json())
    
print(type(countries))
print(countries)

Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [ ]:
# Set the leaders_url variable (1 line)
leaders_url =  "https://country-leaders.onrender.com/leaders"
# query the /leaders endpoint, assign the output to the leaders variable (1 line)
with requests.Session() as s:
    s1 = s.get(cookies_url)
    print(s1.json())
    
    ## checks the cookie
    check = s.get("https://country-leaders.onrender.com/check")
    print(check.json())
    
    s2 = s.get(leaders_url)
    print(s2.json())



# display the leaders variable (1 line)


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [ ]:
# Getting the leaders of a country

# query the /leaders endpoint using cookies and parameters (take any country in countries)
### example url with param: https://country-leaders.onrender.com/leaders?country=ma

with requests.Session() as s:
    params = {"country": "be"}
    s1 = s.get(cookies_url)
    print(s1.json())
    
    ## checks the cookie
    check = s.get("https://country-leaders.onrender.com/check")
    print(check.json())
    
    s2 = s.get(leaders_url, params=params)
    #print(s2.json())
    # assign the output to the leaders variable (1 line)
    leaders = s2.json()
    



# display the leaders variable (1 line)
print(leaders)

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [ ]:
# 4 lines

leaders_by_country = {}
check_url = "https://country-leaders.onrender.com/check"

with requests.Session() as s:
    s1 = s.get(cookies_url)
    print(s.get(check_url).text)

    
    for country in countries:
        #s1 = s.get(cookies_url)
        params = {"country": country}
        print(params)
    
        s2 = s.get(leaders_url, params=params)
        leaders = s2.json()
        leaders_by_country[f"Leaders of {country}"] = leaders
        
    
    

In [ ]:
# or 1 line
print(leaders_by_country)

It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [ ]:
# < 15 lines 
# just a bunch of copy paste it seems
import requests

leaders_by_country = {}

def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    status_url = "https://country-leaders.onrender.com/status"
    countries_url = "https://country-leaders.onrender.com/countries"
    cookies_url = "https://country-leaders.onrender.com/cookie"
    check_url = "https://country-leaders.onrender.com/check"
    leaders_url =  "https://country-leaders.onrender.com/leaders"
    #leaders_by_country = {}
    countries = []
    leaders = []
    

    with requests.Session() as s:
        # getting a ccokie and checking it.    
        s1 = s.get(cookies_url)
        print(s.get(check_url).text)
        
        # Getting the countries
        s2 = s.get(countries_url)
        countries = s2.json()

        ## Getting the leaders of coutries
        for country in countries:
            params = {"country": country}
        
            s3 = s.get(leaders_url, params=params)
            leaders = s3.json()
            leaders_by_country[f"Leaders of {country}"] = leaders
    print("Leaders obtained. See dictionnary leaders_by_country ")

    
            
    return leaders_by_country        
    
get_leaders()
#print(leaders_by_country)


Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [ ]:
# 2 lines

leaders_by_country


## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [77]:
# 3 lines

# Get the page, and gives a header to wikipedia
wiki_url = "https://en.wikipedia.org/wiki/Barack_Obama"
headers = {"User-Agent": "Python exercise, I'll behave!"}
r = requests.get(wiki_url, headers=headers, timeout=10)
print(r.status_code)
#r.text

200


In [ ]:
""" 

# Syntax for using headers in a session
s = requests.Session()
s.headers.update({
    "User-Agent": "MyWikiScraper/1.0 (https://example.com/contact; me@example.com)"
})
r = s.get("https://en.wikipedia.org/wiki/Python_(programming_language)")

 """

Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [ ]:
# 3 lines
from bs4 import BeautifulSoup
soup = BeautifulSoup(r.text, "html.parser")
print(soup.get_text())

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [ ]:
# 2 lines
## Adds all the <p> blocks to the paragraphs list.
paragraphs = []
for par in soup.find_all("p"):
    #paragraphs.append(par.get_text())
    paragraphs.append(par)      
print(paragraphs)

# <p><b>Barack Hussein Obama II</b>

""" 

<p class="mw-empty-elt">
</p>, <p><b>Barack Hussein Obama II</b><sup class="reference" id="cite_ref-2"><a href="#cite_note-2"><span class="cite-bracket">[</span>a<span class="cite-bracket">]</span></a></sup> (born August 4, 1961) is an American politician who served as the 44th <a href="/wiki/President_of_the_United_States" title="President of the United States">president of the United States</a> from 2009 to 2017. A member of the <a href="/wiki/Democratic_Party_(United_States)" title="Democratic Party (United States)">Democratic Party</a>, he was the first <a href="/wiki/African_heritage_of_presidents_of_the_United_States#Barack_Obama" title="African heritage of presidents of the United States">African American president</a>. Obama previously served as a <a class="mw-redirect" href="/wiki/U.S._senator" title="U.S. senator">U.S. senator</a> representing <a href="/wiki/Illinois" title="Illinois">Illinois</a> from 2005 to 2008 and as an <a class="mw-redirect" href="/wiki/Illinois_state_senator" title="Illinois state senator">Illinois state senator</a> from 1997 to 2004.

"""



If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [82]:
# <10 lines
# We have to find the right paragraph:
## Let's try to get a paragraph that start a <b>bold</b> line

# We loop in a list of paragraph, with all the ugly html included.
for par in paragraphs:
    # if the paragraph is empty, skip it
    if not par.get_text(strip=True):
        continue
    
    # find a paragraph that begins with some bold
    first_bold = par.find("b")
    leader_name = first_bold.get_text # putting it in a var for later printing.
    
    # If the first paragraph is not empty, print its text and stop.
    if first_bold:
        print(par.get_text())
        break
        




Barack Hussein Obama II
Barack Hussein Obama II[a] (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.



At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [90]:
# 10 lines
wikipedia_url = "https://en.wikipedia.org/wiki/Barack_Obama"

def get_first_paragraph(wikipedia_url):
  #### 
  # 1/ Get a wikipedia page 
  # 2/ Get the paragraphs > put them in a list
  # 3/ Get the first bold Paragraph > print it 
  # ####
  
  # 1/ Making a request, checking it all works:
  ## we print the url we'll use:
  print(wikipedia_url) # keep this for the rest of the notebook
  
  ## Setting headers, making the request, printing status code:  
  headers = {"User-Agent": "Python exercise, I'll behave!"}
  r = requests.get(wikipedia_url, headers=headers, timeout=10)
  # print(r.status_code) : checks for status, reactivate to debug
  
  # 2/ Retrieving the text from the request:
  ## Create a Beautifulsoup object, a empty paragraphs list.
  soup = BeautifulSoup(r.text, "html.parser")
  paragraphs = []
  
  ## Fills up the paragrapgh list with all the paragraphs
  for par in soup.find_all("p"):
      #paragraphs.append(par.get_text())
      paragraphs.append(par)  
  
  # Filtering for the first paragraph that start with bold:    
  ## Looping though our paragraphs:
  for par in paragraphs:
    if not par.get_text(strip=True):   # if the paragraph is empty, skip it
        continue
    
    ### find a paragraph that begins with some bold
    first_bold = par.find("b")
    leader_name = first_bold.get_text # putting it in a var for later printing.
    
    ### If the first paragraph is not empty, print its text and stop.
    if first_bold:
        print(par.get_text())
        break    

   


   
get_first_paragraph(wikipedia_url)

https://en.wikipedia.org/wiki/Barack_Obama
Barack Hussein Obama II[a] (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.



In [89]:
# Test: 3 lines
get_first_paragraph("https://en.wikipedia.org/wiki/Brad_Pitt")
get_first_paragraph("https://en.wikipedia.org/wiki/Machine_learning")

https://en.wikipedia.org/wiki/Brad_Pitt
200
William Bradley  Pitt (born December 18, 1963) is an American actor and film producer. In a film career spanning more than thirty years, Pitt has received numerous accolades, including two Academy Awards, two British Academy Film Awards, two Golden Globe Awards, two Primetime Emmy Awards and one Volpi Cup. His films as a leading actor have grossed over $7.5 billion worldwide.[3][4][5]

https://en.wikipedia.org/wiki/Machine_learning
200
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions.[1] Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.



### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [ ]:
# 3 lines



Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [ ]:
# 10 lines


Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [ ]:
# < 20 lines


## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [ ]:
# < 20 lines

In [ ]:
# Check the output of your function (2 lines)


Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [ ]:
# < 25 lines



Check the output of your function again.

In [ ]:
# Check the output of your function (1 line)


Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [ ]:
# < 20 lines


Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [ ]:
# <25 lines


Test your new functions.



## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [ ]:
# 3 lines


Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [ ]:
# 3 lines


Make a function `save(leaders_per_country)` to call this code easily.

In [ ]:
# 3 lines


In [ ]:
# Call the function (1 line)


## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!